# Stock histórico y consumo mensual por material/centro

| Fuente | Dataset | Estado |
|---|---|---|
| OData SAP (`ZZ1_BI_MM_STOCK_Q_CDS`) | Stock al 1° de cada mes | Pendiente acceso OData |
| Athena (`sap_pp_docordpro`) | Consumo mensual (mov. 261) | Pendiente que agreguen mov. 261 |

## 1. Setup

In [ ]:
import boto3, time, requests
from requests.auth import HTTPBasicAuth
from dotenv import dotenv_values
from datetime import date
import calendar
import pandas as pd

config = dotenv_values(".env")

AUTH = HTTPBasicAuth(config["SAP_USER"], config["SAP_PASSWORD"])

athena = boto3.client(
    "athena",
    region_name=config["AWS_REGION_NAME"],
    aws_access_key_id=config["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=config["AWS_SECRET_ACCESS_KEY"],
    aws_session_token=config["AWS_SESSION_TOKEN"],
)
ATHENA_DB = "molinos_datalake_datapreparation"
ATHENA_OUTPUT = config["ATHENA_OUTPUT_LOCATION"]

print("Config cargada. SAP user:", config.get("SAP_USER"))

## 2. Período de análisis

In [ ]:
MESES_HISTORIA = 24

def generar_primeros_de_mes(n_meses: int) -> list:
    """Últimos n_meses primeros días de mes, de más antiguo a más reciente."""
    hoy = date.today()
    y, m = hoy.year, hoy.month
    fechas = []
    for _ in range(n_meses):
        m -= 1
        if m == 0:
            m = 12
            y -= 1
        fechas.append(date(y, m, 1))
    return list(reversed(fechas))

fechas = generar_primeros_de_mes(MESES_HISTORIA)
fecha_inicio = fechas[0]
fecha_fin    = fechas[-1]
print(f"Período: {fecha_inicio} → {fecha_fin} ({len(fechas)} meses)")

## 3. Stock al 1° de cada mes — OData SAP

> **Pendiente:** acceso al servicio `ZZ1_BI_MM_STOCK_Q_CDS_0001` (incidente levantado a Basis).
> Correr la celda de test primero, luego la extracción completa.

In [ ]:
STOCK_URL = "https://fiori.mrp.com.ar/sap/opu/odata/sap/ZZ1_BI_MM_STOCK_Q_CDS/ZZ1_BI_MM_STOCK_Q"
STOCK_FIELDS = ["Material", "Plant", "MatlDocLatestPostgDate", "MatlWrhsStkQtyInMatlBaseUnit", "MaterialBaseUnit"]
PAGE_SIZE = 5000

def test_stock(fecha: date):
    """Prueba de conectividad y formato de fecha. Correr antes de la extracción completa."""
    fecha_str = fecha.strftime("%Y%m%d")
    params = {
        "$select": ",".join(STOCK_FIELDS),
        "$filter": f"MatlDocLatestPostgDate eq '{fecha_str}'",
        "$format": "json",
        "$top": "10",
    }
    resp = requests.get(STOCK_URL, params=params, auth=AUTH, headers={"Accept": "application/json"}, timeout=30)
    print("Status:", resp.status_code)
    print("URL:", resp.url)
    print(resp.text[:2000])

# El stock del 1° de cada mes equivale al snapshot del último día del mes anterior
def ultimo_dia_mes_anterior(primera_del_mes: date) -> date:
    y, m = primera_del_mes.year, primera_del_mes.month
    m -= 1
    if m == 0:
        m = 12
        y -= 1
    return date(y, m, calendar.monthrange(y, m)[1])

# Test con la fecha más reciente
test_stock(ultimo_dia_mes_anterior(fechas[-1]))

In [ ]:
def extraer_stock_fecha(primera_del_mes: date) -> pd.DataFrame:
    """Extrae el stock del último día del mes anterior (= foto del 1° del mes)."""
    snapshot = ultimo_dia_mes_anterior(primera_del_mes)
    fecha_str = snapshot.strftime("%Y%m%d")
    filas = []
    skip = 0
    while True:
        params = {
            "$select": ",".join(STOCK_FIELDS),
            "$filter": f"MatlDocLatestPostgDate eq '{fecha_str}'",
            "$format": "json",
            "$top": str(PAGE_SIZE),
            "$skip": str(skip),
        }
        resp = requests.get(STOCK_URL, params=params, auth=AUTH, headers={"Accept": "application/json"})
        resp.raise_for_status()
        data = resp.json()["d"]["results"]
        if not data:
            break
        filas.extend(data)
        if len(data) < PAGE_SIZE:
            break
        skip += PAGE_SIZE
        time.sleep(0.2)
    df = pd.DataFrame(filas).drop(columns=["__metadata"], errors="ignore")
    df["mes"] = primera_del_mes
    return df


def extraer_stock_historico(fechas: list) -> pd.DataFrame:
    dfs = []
    for f in fechas:
        print(f"Stock {f} ...", end=" ")
        df = extraer_stock_fecha(f)
        print(f"{len(df)} filas")
        dfs.append(df)
    historico = pd.concat(dfs, ignore_index=True)
    return historico.rename(columns={
        "Material": "material",
        "Plant": "centro",
        "MatlDocLatestPostgDate": "fecha_snapshot",
        "MatlWrhsStkQtyInMatlBaseUnit": "cantidad_stock",
        "MaterialBaseUnit": "unidad",
    }).assign(cantidad_stock=lambda d: pd.to_numeric(d["cantidad_stock"], errors="coerce"))


df_stock = extraer_stock_historico(fechas)
print(f"\nTotal: {len(df_stock):,} filas | {df_stock['mes'].nunique()} meses | {df_stock['material'].nunique()} materiales")
df_stock.head()

## 4. Consumo mensual — Athena (`sap_pp_docordpro`)

> **Pendiente:** que el equipo de datos agregue el movimiento `261` a la tabla `sap_pp_docordpro`.
> La consulta ya está lista, correr cuando esté disponible.

In [ ]:
def run_athena(sql: str) -> pd.DataFrame:
    r = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": ATHENA_DB},
        ResultConfiguration={"OutputLocation": ATHENA_OUTPUT},
    )
    qid = r["QueryExecutionId"]
    while True:
        status = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
        if status in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(1)
    if status != "SUCCEEDED":
        raise Exception(athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"])
    rows = athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"]
    headers = [d["VarCharValue"] for d in rows[0]["Data"]]
    data = [[d.get("VarCharValue", "") for d in r["Data"]] for r in rows[1:]]
    return pd.DataFrame(data, columns=headers)

In [ ]:
df_consumo = run_athena(f"""
    SELECT
        sku                                              AS material,
        planta                                           AS centro,
        DATE_TRUNC('month', fecha_contabilizacion)       AS mes,
        unidad_base_material                             AS unidad,
        SUM(cantidad_produccion_umb)                     AS cantidad_consumo
    FROM sap_pp_docordpro
    WHERE codigo_tipo_movimiento_mercancia = '261'
      AND fecha_contabilizacion >= DATE '{fecha_inicio}'
      AND fecha_contabilizacion <= DATE '{fecha_fin}'
    GROUP BY
        sku,
        planta,
        DATE_TRUNC('month', fecha_contabilizacion),
        unidad_base_material
    ORDER BY material, centro, mes
""")

df_consumo["mes"] = pd.to_datetime(df_consumo["mes"]).dt.date
df_consumo["cantidad_consumo"] = pd.to_numeric(df_consumo["cantidad_consumo"], errors="coerce")

print(f"Total: {len(df_consumo):,} filas | {df_consumo['mes'].nunique()} meses | {df_consumo['material'].nunique()} materiales")
df_consumo.head()

## 5. Consolidación y salida

In [ ]:
df_final = df_stock[["material", "centro", "mes", "cantidad_stock", "unidad"]].merge(
    df_consumo[["material", "centro", "mes", "cantidad_consumo"]],
    on=["material", "centro", "mes"],
    how="outer",
)

df_final = df_final.sort_values(["material", "centro", "mes"]).reset_index(drop=True)

out_path = "stock_consumo_mensual.csv"
df_final.to_csv(out_path, index=False)
print(f"Listo. {len(df_final):,} filas guardadas en {out_path}")
df_final.head(10)